# Notebook 07 — Exploratory Data Analysis and Predictor Relationships

## Objective

Notebook 07 performs exploratory analysis of the harmonised, study-area-masked,
and year-specific predictor datasets prepared in Notebook 06.

The purpose of this notebook is to understand the statistical distributions,
temporal variations, spatial relationships, and interrelationships among the
predictor variables before developing the flood-susceptibility model.

The analysis uses the three study years:

- 2003
- 2014
- 2025

The predictor datasets contain static and time-varying environmental factors
relevant to flood susceptibility.

### Predictor Groups

**Static predictors**

- Elevation
- Slope
- Flow Accumulation
- River Distance
- Drainage Density
- Clay
- Sand

**Time-varying predictors**

- LULC
- Monsoon Rainfall

### Main Objectives

1. Examine the statistical distributions of continuous predictors.
2. Characterise the composition and temporal change of LULC.
3. Compare monsoon rainfall characteristics between the study years.
4. Examine relationships among continuous predictors.
5. Identify potential predictor redundancy and multicollinearity.
6. Investigate relationships between LULC and continuous environmental
   predictors.
7. Analyse temporal changes in relevant predictor variables.
8. Identify important patterns that should be considered during subsequent
   flood-susceptibility modelling.

### Temporal Comparison Consideration

The year-specific predictor tables do not contain exactly the same number of
valid cells because the availability of some time-varying predictors,
particularly LULC, differs between years.

Therefore, temporal comparisons will distinguish between:

- statistics calculated from each year's available valid observations, and
- comparisons requiring a common spatial sample across years.

No missing predictor values will be artificially filled for exploratory
analysis.

### Methodological Principle

Notebook 07 is exploratory rather than predictive.

The analysis will be used to understand the data and inform the modelling
strategy in Notebook 08. Predictor importance, model performance, and final
flood-susceptibility classification will not be determined solely from
exploratory relationships.

All analyses will use the final predictor tables generated in Notebook 06.

### Expected Outputs

The notebook will produce:

- descriptive-statistics tables,
- predictor distribution plots,
- LULC composition and temporal-change analysis,
- rainfall comparison statistics and plots,
- predictor correlation analysis,
- multicollinearity diagnostics,
- LULC–predictor relationship analysis,
- temporal comparison results,
- and figures suitable for research interpretation and reporting.

All derived tables and figures will be saved separately from the source
predictor datasets.

**Notebook 07 begins the analytical phase of the project and provides the
statistical and spatial evidence required before flood-susceptibility
modelling.**

## 7.1 — Load and Verify Year-Specific Predictor Tables

The year-specific predictor tables generated in Notebook 06 are loaded for
exploratory analysis.

Each table represents the spatially integrated predictors available for one
study year:

- 2003
- 2014
- 2025

The tables contain the seven static predictors together with the
year-specific LULC and monsoon-rainfall variables.

Before beginning the exploratory analysis, the tables are verified for:

- expected columns,
- year identification,
- row counts,
- missing values,
- infinite values,
- duplicate spatial cells,
- and valid LULC class values.

This verification confirms that Notebook 07 is using the final predictor
tables produced by Notebook 06 without modifying them.

No spatial transformation, resampling, interpolation, or missing-value
imputation is performed in this step.

In [1]:
# ============================================================
# Step 07.1 — Load and Verify Year-Specific Predictor Tables
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent


# ============================================================
# PREDICTOR TABLE DIRECTORY
# ============================================================

predictor_table_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "predictors_table"
)


# ============================================================
# EXPECTED FILES
# ============================================================

predictor_paths = {

    2003: (
        predictor_table_dir
        / "predictors_2003_250m.csv"
    ),

    2014: (
        predictor_table_dir
        / "predictors_2014_250m.csv"
    ),

    2025: (
        predictor_table_dir
        / "predictors_2025_250m.csv"
    )
}


# ============================================================
# EXPECTED STRUCTURE
# ============================================================

expected_columns = [
    "year",
    "row",
    "col",
    "x_utm",
    "y_utm",
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand",
    "Rainfall",
    "LULC"
]

expected_lulc_classes = {
    1,
    2,
    3,
    4,
    5
}


expected_row_counts = {
    2003: 47523,
    2014: 46391,
    2025: 43076
}


# ============================================================
# LOAD TABLES
# ============================================================

predictor_tables = {}


for year, path in predictor_paths.items():

    print("\n" + "-" * 75)
    print(f"LOADING PREDICTOR TABLE — {year}")
    print("-" * 75)

    if not path.exists():

        raise FileNotFoundError(
            f"Predictor table not found:\n{path}"
        )

    df = pd.read_csv(path)

    predictor_tables[year] = df

    print(
        f"File       : {path.name}"
    )

    print(
        f"Rows       : {len(df):,}"
    )

    print(
        f"Columns    : {len(df.columns)}"
    )


# ============================================================
# STRUCTURAL VERIFICATION
# ============================================================

print("\n" + "=" * 75)
print("STRUCTURAL VERIFICATION")
print("=" * 75)


for year, df in predictor_tables.items():

    print("\n" + "-" * 75)
    print(f"YEAR — {year}")
    print("-" * 75)


    # --------------------------------------------------------
    # Row count
    # --------------------------------------------------------

    actual_rows = len(df)

    expected_rows = expected_row_counts[year]

    if actual_rows == expected_rows:

        print(
            f"Rows              : "
            f"{actual_rows:,} "
            f"(expected {expected_rows:,}) ✓"
        )

    else:

        raise ValueError(
            f"{year}: expected {expected_rows:,} rows "
            f"but found {actual_rows:,}."
        )


    # --------------------------------------------------------
    # Column structure
    # --------------------------------------------------------

    if list(df.columns) == expected_columns:

        print(
            "Column structure   : ✓"
        )

    else:

        print(
            "Actual columns:"
        )

        print(
            list(df.columns)
        )

        raise ValueError(
            f"{year}: predictor table column structure "
            "does not match the expected structure."
        )


    # --------------------------------------------------------
    # Year identifier
    # --------------------------------------------------------

    unique_years = (
        df["year"]
        .dropna()
        .unique()
    )

    if (
        len(unique_years) == 1
        and
        int(unique_years[0]) == year
    ):

        print(
            f"Year identifier     : {year} ✓"
        )

    else:

        raise ValueError(
            f"{year}: incorrect year identifier."
        )


    # --------------------------------------------------------
    # Missing values
    # --------------------------------------------------------

    missing_count = int(
        df.isna().sum().sum()
    )

    if missing_count == 0:

        print(
            "Missing values      : 0 ✓"
        )

    else:

        raise ValueError(
            f"{year}: found {missing_count:,} "
            "missing values."
        )


    # --------------------------------------------------------
    # Infinite values
    # --------------------------------------------------------

    numeric_columns = (
        df.select_dtypes(
            include=np.number
        ).columns
    )

    infinite_count = int(
        np.isinf(
            df[numeric_columns].to_numpy()
        ).sum()
    )

    if infinite_count == 0:

        print(
            "Infinite values     : 0 ✓"
        )

    else:

        raise ValueError(
            f"{year}: found {infinite_count:,} "
            "infinite values."
        )


    # --------------------------------------------------------
    # Duplicate spatial cells
    # --------------------------------------------------------

    duplicate_cells = int(
        df.duplicated(
            subset=["row", "col"]
        ).sum()
    )

    if duplicate_cells == 0:

        print(
            "Duplicate cells     : 0 ✓"
        )

    else:

        raise ValueError(
            f"{year}: found {duplicate_cells:,} "
            "duplicate spatial cells."
        )


    # --------------------------------------------------------
    # Coordinate validity
    # --------------------------------------------------------

    if (
        np.isfinite(
            df["x_utm"]
        ).all()
        and
        np.isfinite(
            df["y_utm"]
        ).all()
    ):

        print(
            "Coordinates         : ✓"
        )

    else:

        raise ValueError(
            f"{year}: invalid UTM coordinates detected."
        )


    # --------------------------------------------------------
    # LULC classes
    # --------------------------------------------------------

    lulc_classes = set(
        df["LULC"]
        .astype(int)
        .unique()
    )

    print(
        f"LULC classes        : "
        f"{sorted(lulc_classes)}"
    )

    if lulc_classes == expected_lulc_classes:

        print(
            "LULC class validity : ✓"
        )

    else:

        raise ValueError(
            f"{year}: unexpected LULC classes "
            f"detected: {sorted(lulc_classes)}"
        )


# ============================================================
# DISPLAY TABLE STRUCTURE
# ============================================================

print("\n" + "=" * 75)
print("PREDICTOR TABLE STRUCTURE")
print("=" * 75)

for year, df in predictor_tables.items():

    print("\n" + "-" * 75)
    print(f"{year}")
    print("-" * 75)

    print(
        df.dtypes.to_string()
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.1 — FINAL STATUS")
print("=" * 75)

print(
    "✓ 2003 predictor table loaded"
)

print(
    "✓ 2014 predictor table loaded"
)

print(
    "✓ 2025 predictor table loaded"
)

print(
    "✓ Expected column structure confirmed"
)

print(
    "✓ Row counts confirmed"
)

print(
    "✓ Year identifiers confirmed"
)

print(
    "✓ No missing values detected"
)

print(
    "✓ No infinite values detected"
)

print(
    "✓ No duplicate spatial cells detected"
)

print(
    "✓ UTM coordinates are valid"
)

print(
    "✓ LULC classes are valid"
)

print(
    "✓ Source predictor tables were not modified"
)

print("=" * 75)
print("✓ STEP 07.1 COMPLETED")
print("=" * 75)


---------------------------------------------------------------------------
LOADING PREDICTOR TABLE — 2003
---------------------------------------------------------------------------
File       : predictors_2003_250m.csv
Rows       : 47,523
Columns    : 14

---------------------------------------------------------------------------
LOADING PREDICTOR TABLE — 2014
---------------------------------------------------------------------------
File       : predictors_2014_250m.csv
Rows       : 46,391
Columns    : 14

---------------------------------------------------------------------------
LOADING PREDICTOR TABLE — 2025
---------------------------------------------------------------------------
File       : predictors_2025_250m.csv
Rows       : 43,076
Columns    : 14

STRUCTURAL VERIFICATION

---------------------------------------------------------------------------
YEAR — 2003
---------------------------------------------------------------------------
Rows              : 47,523 (expected

## 7.2 — Descriptive Statistics of Continuous Predictors

Descriptive statistics are calculated for the continuous environmental
predictors in the year-specific predictor tables.

The analysis includes:

- Elevation
- Slope
- Flow Accumulation
- River Distance
- Drainage Density
- Clay
- Sand
- Monsoon Rainfall

For each predictor and study year, the following statistics are calculated:

- Number of observations
- Mean
- Standard deviation
- Minimum
- 25th percentile
- Median
- 75th percentile
- Maximum

Identifier fields, spatial coordinates, year identifiers, and the categorical
LULC variable are excluded from this analysis.

The statistics are calculated independently for the valid observations
available in each year's predictor table. No missing values are imputed and
no observations are artificially added.

The resulting statistics provide the baseline distributional characteristics
of the predictors and help identify potential skewness, extreme values,
temporal differences, and variables that may require additional investigation
before flood-susceptibility modelling.

The resulting statistical table is saved separately as a derived output and
does not modify the original predictor tables.

In [2]:
# ============================================================
# Step 07.2 — Descriptive Statistics of Continuous Predictors
# ============================================================

# ============================================================
# CONTINUOUS PREDICTORS
# ============================================================

continuous_predictors = [
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand",
    "Rainfall"
]


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

statistics_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
)

statistics_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CALCULATE DESCRIPTIVE STATISTICS
# ============================================================

statistics_records = []


for year, df in predictor_tables.items():

    print("\n" + "-" * 75)
    print(f"YEAR — {year}")
    print("-" * 75)

    for predictor in continuous_predictors:

        values = (
            pd.to_numeric(
                df[predictor],
                errors="coerce"
            )
        )

        # ----------------------------------------------------
        # Verify valid numerical values
        # ----------------------------------------------------

        if values.isna().any():

            raise ValueError(
                f"{year} — {predictor}: "
                "missing or non-numeric values detected."
            )

        if not np.isfinite(
            values.to_numpy()
        ).all():

            raise ValueError(
                f"{year} — {predictor}: "
                "infinite values detected."
            )

        # ----------------------------------------------------
        # Calculate statistics
        # ----------------------------------------------------

        record = {

            "year": year,

            "predictor": predictor,

            "count": int(
                values.count()
            ),

            "mean": float(
                values.mean()
            ),

            "std": float(
                values.std(
                    ddof=1
                )
            ),

            "minimum": float(
                values.min()
            ),

            "q25": float(
                values.quantile(0.25)
            ),

            "median": float(
                values.median()
            ),

            "q75": float(
                values.quantile(0.75)
            ),

            "maximum": float(
                values.max()
            )
        }

        statistics_records.append(
            record
        )

        print(
            f"{predictor:<20}"
            f"mean={record['mean']:.4f}  "
            f"median={record['median']:.4f}  "
            f"min={record['minimum']:.4f}  "
            f"max={record['maximum']:.4f}"
        )


# ============================================================
# CREATE STATISTICS TABLE
# ============================================================

descriptive_statistics = pd.DataFrame(
    statistics_records
)


# ============================================================
# VERIFY EXPECTED RECORD COUNT
# ============================================================

expected_records = (
    len(predictor_tables)
    * len(continuous_predictors)
)

if len(descriptive_statistics) != expected_records:

    raise ValueError(
        "Unexpected number of descriptive-statistics records."
    )


# ============================================================
# SAVE TABLE
# ============================================================

statistics_path = (
    statistics_dir
    / "continuous_predictor_descriptive_statistics_2003_2014_2025.csv"
)

descriptive_statistics.to_csv(
    statistics_path,
    index=False
)


# ============================================================
# FINAL TABLE PREVIEW
# ============================================================

print("\n" + "=" * 75)
print("DESCRIPTIVE STATISTICS TABLE")
print("=" * 75)

print(
    descriptive_statistics.to_string(
        index=False
    )
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.2 — FINAL STATUS")
print("=" * 75)

print(
    f"✓ Years analysed       : "
    f"{len(predictor_tables)}"
)

print(
    f"✓ Continuous predictors: "
    f"{len(continuous_predictors)}"
)

print(
    f"✓ Statistics generated : "
    f"{len(descriptive_statistics)} records"
)

print(
    "✓ No missing values used"
)

print(
    "✓ No infinite values used"
)

print(
    f"✓ Output saved         : "
    f"{statistics_path}"
)

print(
    "✓ Original predictor tables were not modified"
)

print("=" * 75)
print("✓ STEP 07.2 COMPLETED")
print("=" * 75)


---------------------------------------------------------------------------
YEAR — 2003
---------------------------------------------------------------------------
Elevation           mean=120.6940  median=121.0534  min=101.2746  max=135.9356
Slope               mean=1.5668  median=1.4616  min=0.0000  max=8.3203
Flow_Accumulation   mean=3913.0912  median=19.9406  min=2.1022  max=582647.0600
River_Distance      mean=1277.0529  median=1145.2377  min=38.3393  max=5129.2850
Drainage_Density    mean=0.3929  median=0.3023  min=0.0000  max=2.1059
Clay                mean=25.1674  median=26.4161  min=0.0000  max=33.8261
Sand                mean=32.0595  median=34.0676  min=0.0000  max=42.5395
Rainfall            mean=853.9031  median=908.3202  min=155.2107  max=1003.2170

---------------------------------------------------------------------------
YEAR — 2014
---------------------------------------------------------------------------
Elevation           mean=120.6396  median=120.9422  min=101.

## 7.3 — Continuous Predictor Distribution Analysis

The distributions of the continuous environmental predictors are visualised
to complement the descriptive statistics calculated in Step 07.2.

The analysis includes:

- Elevation
- Slope
- Flow Accumulation
- River Distance
- Drainage Density
- Clay
- Sand
- Monsoon Rainfall

Distribution plots are generated separately for the three study years where
the predictor is year-specific. Static predictors are compared across years
using their available observations.

The visualisations are used to identify:

- concentration of observations,
- skewed distributions,
- potential extreme values,
- differences in the range and distribution of predictors,
- and variables that may require additional statistical consideration.

The plots are exploratory and are not used to remove observations or modify
the predictor datasets.

No transformation, standardisation, outlier removal, or missing-value
imputation is performed at this stage.

All figures are saved as derived outputs under:

`outputs/figures/notebook_07/`

In [3]:
# ============================================================
# Step 07.3 — Continuous Predictor Distribution Analysis
# ============================================================
import matplotlib.pyplot as plt


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

distribution_figure_dir = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
    / "notebook_07"
    / "distributions"
)

distribution_figure_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# STATIC CONTINUOUS PREDICTORS
# ============================================================

static_predictors = [
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand"
]


# ============================================================
# PREDICTOR UNITS
# ============================================================

predictor_units = {

    "Elevation": "Elevation (m)",

    "Slope": "Slope (degrees)",

    "Flow_Accumulation":
        "Flow Accumulation",

    "River_Distance":
        "River Distance (m)",

    "Drainage_Density":
        "Drainage Density",

    "Clay":
        "Clay (%)",

    "Sand":
        "Sand (%)",

    "Rainfall":
        "Monsoon Rainfall (mm)"
}


# ============================================================
# 1. STATIC PREDICTOR DISTRIBUTIONS
# ============================================================

print("\n" + "-" * 75)
print("STATIC PREDICTORS")
print("-" * 75)


for predictor in static_predictors:

    print(
        f"Plotting: {predictor}"
    )

    figure = plt.figure(
        figsize=(9, 6)
    )

    axis = figure.add_axes(
        [0.10, 0.12, 0.82, 0.78]
    )

    plotted = False

    for year, df in predictor_tables.items():

        values = (
            df[predictor]
            .to_numpy(dtype=float)
        )

        values = values[
            np.isfinite(values)
        ]

        if values.size == 0:

            raise ValueError(
                f"No valid values found for "
                f"{predictor} in {year}."
            )

        axis.hist(
            values,
            bins=50,
            density=True,
            histtype="step",
            linewidth=1.5,
            label=str(year)
        )

        plotted = True

    if not plotted:

        plt.close(figure)

        raise ValueError(
            f"No distributions plotted for {predictor}."
        )

    axis.set_title(
        f"{predictor} — Distribution"
    )

    axis.set_xlabel(
        predictor_units[predictor]
    )

    axis.set_ylabel(
        "Density"
    )

    axis.legend(
        title="Year"
    )

    axis.grid(
        alpha=0.25
    )

    figure_path = (
        distribution_figure_dir
        / f"{predictor.replace(' ', '_')}_distribution.png"
    )

    figure.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(
        figure
    )


# ============================================================
# 2. YEAR-SPECIFIC RAINFALL DISTRIBUTIONS
# ============================================================

print("\n" + "-" * 75)
print("RAINFALL")
print("-" * 75)


figure = plt.figure(
    figsize=(9, 6)
)

axis = figure.add_axes(
    [0.10, 0.12, 0.82, 0.78]
)


for year, df in predictor_tables.items():

    values = (
        df["Rainfall"]
        .to_numpy(dtype=float)
    )

    values = values[
        np.isfinite(values)
    ]

    if values.size == 0:

        raise ValueError(
            f"No valid rainfall values found "
            f"for {year}."
        )

    axis.hist(
        values,
        bins=50,
        density=True,
        histtype="step",
        linewidth=1.5,
        label=str(year)
    )


axis.set_title(
    "Monsoon Rainfall — Distribution"
)

axis.set_xlabel(
    predictor_units["Rainfall"]
)

axis.set_ylabel(
    "Density"
)

axis.legend(
    title="Year"
)

axis.grid(
    alpha=0.25
)


rainfall_figure_path = (
    distribution_figure_dir
    / "Rainfall_distribution_2003_2014_2025.png"
)

figure.savefig(
    rainfall_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(
    figure
)


# ============================================================
# 3. FIGURE INVENTORY
# ============================================================

print("\n" + "-" * 75)
print("FIGURE OUTPUTS")
print("-" * 75)


figure_files = sorted(
    distribution_figure_dir.glob("*.png")
)


for figure_path in figure_files:

    size_mb = (
        figure_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"✓ {figure_path.name:<60}"
        f"{size_mb:.2f} MB"
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.3 — FINAL STATUS")
print("=" * 75)

print(
    f"✓ Static predictors plotted : "
    f"{len(static_predictors)}"
)

print(
    "✓ Rainfall distributions plotted"
)

print(
    f"✓ Total figures generated   : "
    f"{len(figure_files)}"
)

print(
    f"✓ Output directory           : "
    f"{distribution_figure_dir}"
)

print(
    "✓ No observations removed"
)

print(
    "✓ No predictor values modified"
)

print(
    "✓ No transformation performed"
)

print("=" * 75)
print("✓ STEP 07.3 COMPLETED")
print("=" * 75)


---------------------------------------------------------------------------
STATIC PREDICTORS
---------------------------------------------------------------------------
Plotting: Elevation
Plotting: Slope
Plotting: Flow_Accumulation
Plotting: River_Distance
Plotting: Drainage_Density
Plotting: Clay
Plotting: Sand

---------------------------------------------------------------------------
RAINFALL
---------------------------------------------------------------------------

---------------------------------------------------------------------------
FIGURE OUTPUTS
---------------------------------------------------------------------------
✓ Clay_distribution.png                                       0.08 MB
✓ Drainage_Density_distribution.png                           0.07 MB
✓ Elevation_distribution.png                                  0.08 MB
✓ Flow_Accumulation_distribution.png                          0.07 MB
✓ Rainfall_distribution_2003_2014_2025.png                    0.08 MB
✓ R

## 7.4 — LULC Composition and Temporal Change

The temporal composition of land-use/land-cover (LULC) classes is analysed
for 2003, 2014, and 2025.

The analysis uses the harmonised 250 m LULC datasets generated in Notebook 06
and examines the distribution of the five categorical classes:

1. Water
2. Vegetation
3. Built-up
4. Barren
5. Agriculture

For each study year, the following quantities are calculated:

- Number of valid cells in each LULC class
- Percentage of valid LULC cells represented by each class
- Approximate area represented by each class
- Temporal changes in class area
- Temporal changes in class percentage share

The analysis treats LULC as a categorical variable. Class identifiers are not
interpreted as continuous numerical measurements.

Because the number of valid LULC cells differs between study years, class
percentages are calculated relative to the valid LULC cells available in each
year. Therefore, differences in spatial coverage are retained and are not
artificially filled or corrected.

The analysis is exploratory and is intended to quantify the temporal
landscape changes that may influence flood susceptibility.

No LULC values are modified, reclassified, interpolated, or imputed during
this step.

Derived statistics are saved under:

`outputs/tables/notebook_07/`

The temporal composition results will later be considered alongside the
continuous predictor relationships and flood-susceptibility modelling.

In [4]:
# ============================================================
# Step 07.4 — LULC Composition and Temporal Change
# ============================================================

import numpy as np
import pandas as pd


print("\n" + "=" * 75)
print("NOTEBOOK 07 — LULC COMPOSITION AND TEMPORAL CHANGE")
print("=" * 75)


# ============================================================
# LULC CLASS DEFINITIONS
# ============================================================

lulc_classes = {
    1: "Water",
    2: "Vegetation",
    3: "Built-up",
    4: "Barren",
    5: "Agriculture"
}


# ============================================================
# AREA PER MODELLING CELL
# ============================================================

CELL_SIZE_M = 250.0

CELL_AREA_M2 = (
    CELL_SIZE_M
    * CELL_SIZE_M
)

CELL_AREA_KM2 = (
    CELL_AREA_M2
    / 1_000_000
)


print("\n" + "-" * 75)
print("MODELLING CELL AREA")
print("-" * 75)

print(
    f"Cell size : "
    f"{CELL_SIZE_M:.0f} × {CELL_SIZE_M:.0f} m"
)

print(
    f"Cell area : "
    f"{CELL_AREA_KM2:.4f} km²"
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

lulc_table_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
)

lulc_table_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CALCULATE LULC COMPOSITION
# ============================================================

lulc_records = []


for year, df in predictor_tables.items():

    print("\n" + "-" * 75)
    print(f"LULC COMPOSITION — {year}")
    print("-" * 75)


    # --------------------------------------------------------
    # Extract valid LULC values
    # --------------------------------------------------------

    lulc = (
        pd.to_numeric(
            df["LULC"],
            errors="coerce"
        )
    )

    lulc = lulc[
        lulc.isin(
            list(lulc_classes.keys())
        )
    ]

    if len(lulc) == 0:

        raise ValueError(
            f"{year}: no valid LULC observations found."
        )


    # --------------------------------------------------------
    # Total valid cells
    # --------------------------------------------------------

    total_valid_cells = len(lulc)


    # --------------------------------------------------------
    # Class statistics
    # --------------------------------------------------------

    for class_code, class_name in lulc_classes.items():

        cell_count = int(
            (lulc == class_code).sum()
        )

        percentage = (
            cell_count
            / total_valid_cells
            * 100
        )

        area_km2 = (
            cell_count
            * CELL_AREA_KM2
        )

        lulc_records.append({

            "year": year,

            "class_code": class_code,

            "class_name": class_name,

            "cell_count": cell_count,

            "percentage": percentage,

            "area_km2": area_km2
        })


        print(
            f"Class {class_code} — "
            f"{class_name:<12}: "
            f"{cell_count:>7,} cells | "
            f"{percentage:>6.2f}% | "
            f"{area_km2:>8.3f} km²"
        )


    print(
        f"Total valid LULC cells: "
        f"{total_valid_cells:,}"
    )


# ============================================================
# CREATE COMPOSITION TABLE
# ============================================================

lulc_composition = pd.DataFrame(
    lulc_records
)


# ============================================================
# VERIFY CLASS COVERAGE
# ============================================================

for year in predictor_tables.keys():

    year_data = (
        lulc_composition[
            lulc_composition["year"] == year
        ]
    )

    percentage_sum = (
        year_data["percentage"].sum()
    )

    if not np.isclose(
        percentage_sum,
        100.0,
        atol=0.01
    ):

        raise ValueError(
            f"{year}: LULC percentages do not "
            f"sum to 100%. "
            f"Obtained {percentage_sum:.4f}%."
        )


# ============================================================
# SAVE COMPOSITION TABLE
# ============================================================

composition_path = (
    lulc_table_dir
    / "LULC_composition_2003_2014_2025.csv"
)

lulc_composition.to_csv(
    composition_path,
    index=False
)


# ============================================================
# TEMPORAL CHANGE — AREA
# ============================================================

area_table = (
    lulc_composition
    .pivot(
        index="class_name",
        columns="year",
        values="area_km2"
    )
)

area_table["change_2003_2014_km2"] = (
    area_table[2014]
    - area_table[2003]
)

area_table["change_2014_2025_km2"] = (
    area_table[2025]
    - area_table[2014]
)

area_table["change_2003_2025_km2"] = (
    area_table[2025]
    - area_table[2003]
)


area_change_path = (
    lulc_table_dir
    / "LULC_area_change_2003_2014_2025.csv"
)

area_table.to_csv(
    area_change_path
)


# ============================================================
# TEMPORAL CHANGE — PERCENTAGE SHARE
# ============================================================

percentage_table = (
    lulc_composition
    .pivot(
        index="class_name",
        columns="year",
        values="percentage"
    )
)

percentage_table["change_2003_2014_percentage_points"] = (
    percentage_table[2014]
    - percentage_table[2003]
)

percentage_table["change_2014_2025_percentage_points"] = (
    percentage_table[2025]
    - percentage_table[2014]
)

percentage_table["change_2003_2025_percentage_points"] = (
    percentage_table[2025]
    - percentage_table[2003]
)


percentage_change_path = (
    lulc_table_dir
    / "LULC_percentage_change_2003_2014_2025.csv"
)

percentage_table.to_csv(
    percentage_change_path
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 75)
print("LULC COMPOSITION TABLE")
print("=" * 75)

print(
    lulc_composition.to_string(
        index=False
    )
)


print("\n" + "=" * 75)
print("LULC AREA CHANGE — km²")
print("=" * 75)

print(
    area_table.to_string()
)


print("\n" + "=" * 75)
print("LULC SHARE CHANGE — PERCENTAGE POINTS")
print("=" * 75)

print(
    percentage_table.to_string()
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.4 — FINAL STATUS")
print("=" * 75)

print(
    "✓ Five LULC classes analysed"
)

print(
    "✓ LULC treated as categorical data"
)

print(
    "✓ Class counts calculated for 2003, 2014, and 2025"
)

print(
    "✓ Class percentages calculated"
)

print(
    "✓ Approximate class areas calculated"
)

print(
    "✓ Temporal area changes calculated"
)

print(
    "✓ Temporal percentage-share changes calculated"
)

print(
    f"✓ Composition table saved: "
    f"{composition_path}"
)

print(
    f"✓ Area-change table saved: "
    f"{area_change_path}"
)

print(
    f"✓ Percentage-change table saved: "
    f"{percentage_change_path}"
)

print(
    "✓ No LULC values were modified"
)

print(
    "✓ No missing values were imputed"
)

print("=" * 75)
print("✓ STEP 07.4 COMPLETED")
print("=" * 75)


NOTEBOOK 07 — LULC COMPOSITION AND TEMPORAL CHANGE

---------------------------------------------------------------------------
MODELLING CELL AREA
---------------------------------------------------------------------------
Cell size : 250 × 250 m
Cell area : 0.0625 km²

---------------------------------------------------------------------------
LULC COMPOSITION — 2003
---------------------------------------------------------------------------
Class 1 — Water       :     166 cells |   0.35% |   10.375 km²
Class 2 — Vegetation  :   5,621 cells |  11.83% |  351.312 km²
Class 3 — Built-up    :   9,177 cells |  19.31% |  573.562 km²
Class 4 — Barren      :  29,694 cells |  62.48% | 1855.875 km²
Class 5 — Agriculture :   2,865 cells |   6.03% |  179.062 km²
Total valid LULC cells: 47,523

---------------------------------------------------------------------------
LULC COMPOSITION — 2014
---------------------------------------------------------------------------
Class 1 — Water       :     

## 7.5 — Monsoon Rainfall Temporal Analysis

The spatial and temporal characteristics of monsoon rainfall are analysed for
2003, 2014, and 2025.

Rainfall is treated as a continuous environmental predictor and is expressed
in millimetres (mm).

For each study year, the following statistics are calculated:

- Number of observations
- Mean
- Standard deviation
- Minimum
- 25th percentile
- Median
- 75th percentile
- Maximum

The analysis also quantifies temporal changes in mean rainfall between:

- 2003 and 2014
- 2014 and 2025
- 2003 and 2025

Both absolute change in millimetres and percentage change are calculated.

Because the rainfall predictor has complete coverage across the study-area
mask for all three years, the rainfall statistics are directly comparable
with respect to spatial coverage.

The analysis is exploratory and is intended to identify differences in the
magnitude and spatial distribution of monsoon rainfall among the study years.

No rainfall values are modified, interpolated, standardised, or imputed during
this step.

Derived statistics and comparison tables are saved separately from the
original predictor datasets.

In [5]:
# ============================================================
# Step 07.5 — Monsoon Rainfall Temporal Analysis
# ============================================================

import numpy as np
import pandas as pd


print("\n" + "=" * 75)
print("NOTEBOOK 07 — MONSOON RAINFALL TEMPORAL ANALYSIS")
print("=" * 75)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

rainfall_table_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
)

rainfall_table_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CALCULATE YEAR-WISE RAINFALL STATISTICS
# ============================================================

rainfall_records = []


for year, df in predictor_tables.items():

    print("\n" + "-" * 75)
    print(f"RAINFALL — {year}")
    print("-" * 75)


    # --------------------------------------------------------
    # Extract rainfall values
    # --------------------------------------------------------

    rainfall = pd.to_numeric(
        df["Rainfall"],
        errors="coerce"
    )


    # --------------------------------------------------------
    # Validate rainfall values
    # --------------------------------------------------------

    if rainfall.isna().any():

        raise ValueError(
            f"{year}: missing rainfall values detected."
        )


    rainfall_values = rainfall.to_numpy(
        dtype=float
    )


    if not np.isfinite(
        rainfall_values
    ).all():

        raise ValueError(
            f"{year}: infinite rainfall values detected."
        )


    if (
        rainfall_values < 0
    ).any():

        raise ValueError(
            f"{year}: negative rainfall values detected."
        )


    # --------------------------------------------------------
    # Calculate statistics
    # --------------------------------------------------------

    record = {

        "year": year,

        "count": int(
            rainfall.count()
        ),

        "mean_mm": float(
            rainfall.mean()
        ),

        "std_mm": float(
            rainfall.std(
                ddof=1
            )
        ),

        "minimum_mm": float(
            rainfall.min()
        ),

        "q25_mm": float(
            rainfall.quantile(0.25)
        ),

        "median_mm": float(
            rainfall.median()
        ),

        "q75_mm": float(
            rainfall.quantile(0.75)
        ),

        "maximum_mm": float(
            rainfall.max()
        )
    }


    rainfall_records.append(
        record
    )


    # --------------------------------------------------------
    # Display statistics
    # --------------------------------------------------------

    print(
        f"Observations : "
        f"{record['count']:,}"
    )

    print(
        f"Mean         : "
        f"{record['mean_mm']:.2f} mm"
    )

    print(
        f"Std. Dev.    : "
        f"{record['std_mm']:.2f} mm"
    )

    print(
        f"Minimum      : "
        f"{record['minimum_mm']:.2f} mm"
    )

    print(
        f"Q25          : "
        f"{record['q25_mm']:.2f} mm"
    )

    print(
        f"Median       : "
        f"{record['median_mm']:.2f} mm"
    )

    print(
        f"Q75          : "
        f"{record['q75_mm']:.2f} mm"
    )

    print(
        f"Maximum      : "
        f"{record['maximum_mm']:.2f} mm"
    )


# ============================================================
# CREATE STATISTICS TABLE
# ============================================================

rainfall_statistics = pd.DataFrame(
    rainfall_records
)


# ============================================================
# VERIFY ALL YEARS ARE PRESENT
# ============================================================

expected_years = {
    2003,
    2014,
    2025
}

actual_years = set(
    rainfall_statistics["year"]
)


if actual_years != expected_years:

    raise ValueError(
        "Rainfall statistics do not contain "
        "the expected study years."
    )


# ============================================================
# SAVE STATISTICS TABLE
# ============================================================

rainfall_statistics_path = (
    rainfall_table_dir
    / "rainfall_statistics_2003_2014_2025.csv"
)

rainfall_statistics.to_csv(
    rainfall_statistics_path,
    index=False
)


# ============================================================
# MEAN RAINFALL TEMPORAL CHANGE
# ============================================================

mean_rainfall = (
    rainfall_statistics
    .set_index("year")["mean_mm"]
)


rainfall_change_records = [

    {
        "comparison": "2003_to_2014",

        "initial_year": 2003,

        "final_year": 2014,

        "initial_mean_mm":
            float(mean_rainfall[2003]),

        "final_mean_mm":
            float(mean_rainfall[2014]),

        "absolute_change_mm":
            float(
                mean_rainfall[2014]
                - mean_rainfall[2003]
            ),

        "percentage_change":
            float(
                (
                    (
                        mean_rainfall[2014]
                        - mean_rainfall[2003]
                    )
                    / mean_rainfall[2003]
                )
                * 100
            )
    },

    {
        "comparison": "2014_to_2025",

        "initial_year": 2014,

        "final_year": 2025,

        "initial_mean_mm":
            float(mean_rainfall[2014]),

        "final_mean_mm":
            float(mean_rainfall[2025]),

        "absolute_change_mm":
            float(
                mean_rainfall[2025]
                - mean_rainfall[2014]
            ),

        "percentage_change":
            float(
                (
                    (
                        mean_rainfall[2025]
                        - mean_rainfall[2014]
                    )
                    / mean_rainfall[2014]
                )
                * 100
            )
    },

    {
        "comparison": "2003_to_2025",

        "initial_year": 2003,

        "final_year": 2025,

        "initial_mean_mm":
            float(mean_rainfall[2003]),

        "final_mean_mm":
            float(mean_rainfall[2025]),

        "absolute_change_mm":
            float(
                mean_rainfall[2025]
                - mean_rainfall[2003]
            ),

        "percentage_change":
            float(
                (
                    (
                        mean_rainfall[2025]
                        - mean_rainfall[2003]
                    )
                    / mean_rainfall[2003]
                )
                * 100
            )
    }
]


rainfall_change = pd.DataFrame(
    rainfall_change_records
)


# ============================================================
# SAVE TEMPORAL CHANGE TABLE
# ============================================================

rainfall_change_path = (
    rainfall_table_dir
    / "rainfall_mean_temporal_change_2003_2014_2025.csv"
)

rainfall_change.to_csv(
    rainfall_change_path,
    index=False
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 75)
print("RAINFALL STATISTICS")
print("=" * 75)

print(
    rainfall_statistics.to_string(
        index=False
    )
)


print("\n" + "=" * 75)
print("MEAN RAINFALL TEMPORAL CHANGE")
print("=" * 75)

print(
    rainfall_change.to_string(
        index=False
    )
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.5 — FINAL STATUS")
print("=" * 75)

print(
    "✓ Rainfall analysed for 2003, 2014, and 2025"
)

print(
    "✓ Rainfall treated as a continuous predictor"
)

print(
    "✓ Year-wise descriptive statistics calculated"
)

print(
    "✓ Mean rainfall temporal changes calculated"
)

print(
    "✓ Absolute changes calculated"
)

print(
    "✓ Percentage changes calculated"
)

print(
    f"✓ Statistics table saved: "
    f"{rainfall_statistics_path}"
)

print(
    f"✓ Temporal-change table saved: "
    f"{rainfall_change_path}"
)

print(
    "✓ No rainfall values were modified"
)

print(
    "✓ No missing values were imputed"
)

print("=" * 75)
print("✓ STEP 07.5 COMPLETED")
print("=" * 75)


NOTEBOOK 07 — MONSOON RAINFALL TEMPORAL ANALYSIS

---------------------------------------------------------------------------
RAINFALL — 2003
---------------------------------------------------------------------------
Observations : 47,523
Mean         : 853.90 mm
Std. Dev.    : 137.34 mm
Minimum      : 155.21 mm
Q25          : 847.22 mm
Median       : 908.32 mm
Q75          : 935.98 mm
Maximum      : 1003.22 mm

---------------------------------------------------------------------------
RAINFALL — 2014
---------------------------------------------------------------------------
Observations : 46,391
Mean         : 708.35 mm
Std. Dev.    : 113.33 mm
Minimum      : 115.35 mm
Q25          : 708.22 mm
Median       : 742.81 mm
Q75          : 771.30 mm
Maximum      : 834.59 mm

---------------------------------------------------------------------------
RAINFALL — 2025
---------------------------------------------------------------------------
Observations : 43,076
Mean         : 917.89 mm
S

## 7.6 — Continuous Predictor Correlation Analysis

Pearson correlation analysis is performed to examine linear relationships
among the continuous environmental predictors used in the flood-susceptibility
framework.

The continuous predictors include:

- Elevation
- Slope
- Flow Accumulation
- River Distance
- Drainage Density
- Clay
- Sand
- Monsoon Rainfall

LULC is excluded from Pearson correlation analysis because it represents
categorical land-use/land-cover classes rather than a continuous numerical
variable.

Because rainfall varies between study years while the remaining physical
predictors are treated as static predictors, the analysis is separated into:

1. A static-predictor correlation matrix containing the seven static
   continuous predictors.
2. Year-specific correlation matrices containing the seven static predictors
   and the rainfall predictor for 2003, 2014, and 2025.

Pearson's correlation coefficient (r) ranges from -1 to +1:

- Values close to +1 indicate a strong positive linear relationship.
- Values close to -1 indicate a strong negative linear relationship.
- Values close to 0 indicate a weak or absent linear linear relationship.

The correlation analysis is exploratory and is used to identify potentially
redundant predictors and possible multicollinearity before modelling.

Correlation does not imply causation, and no predictor is removed solely on
the basis of correlation at this stage.

Correlation matrices are saved as CSV files and corresponding heatmaps are
saved as PNG figures.

No predictor values are modified, transformed, standardised, or removed.

In [6]:
# ============================================================
# Step 07.6 — Continuous Predictor Correlation Analysis
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


print("\n" + "=" * 75)
print("NOTEBOOK 07 — CONTINUOUS PREDICTOR CORRELATION ANALYSIS")
print("=" * 75)


# ============================================================
# PREDICTOR DEFINITIONS
# ============================================================

static_predictors = [
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand"
]

rainfall_predictor = "Rainfall"


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

correlation_table_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
    / "correlation"
)

correlation_figure_dir = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
    / "notebook_07"
    / "correlation"
)

correlation_table_dir.mkdir(
    parents=True,
    exist_ok=True
)

correlation_figure_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# HELPER FUNCTION — CORRELATION HEATMAP
# ============================================================

def save_correlation_heatmap(
    correlation_matrix,
    title,
    output_path
):

    figure = plt.figure(
        figsize=(10, 8)
    )

    axis = figure.add_axes(
        [0.12, 0.12, 0.78, 0.76]
    )

    image = axis.imshow(
        correlation_matrix.values,
        vmin=-1,
        vmax=1,
        aspect="auto"
    )

    axis.set_xticks(
        range(
            len(correlation_matrix.columns)
        )
    )

    axis.set_yticks(
        range(
            len(correlation_matrix.index)
        )
    )

    axis.set_xticklabels(
        correlation_matrix.columns,
        rotation=45,
        ha="right"
    )

    axis.set_yticklabels(
        correlation_matrix.index
    )

    axis.set_title(
        title
    )

    # --------------------------------------------------------
    # Add correlation values
    # --------------------------------------------------------

    for row in range(
        len(correlation_matrix.index)
    ):

        for col in range(
            len(correlation_matrix.columns)
        ):

            value = correlation_matrix.iloc[
                row,
                col
            ]

            axis.text(
                col,
                row,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8
            )

    figure.colorbar(
        image,
        ax=axis,
        label="Pearson r",
        fraction=0.046,
        pad=0.04
    )

    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(
        figure
    )


# ============================================================
# 1. STATIC PREDICTOR CORRELATION
# ============================================================

print("\n" + "-" * 75)
print("STATIC PREDICTOR CORRELATION")
print("-" * 75)


# ------------------------------------------------------------
# Use 2003 table for static predictors only.
#
# The physical static predictors are identical across years,
# while their valid-cell masks may differ because of the
# year-specific common masks.
# ------------------------------------------------------------

static_df = predictor_tables[2003][
    static_predictors
].copy()


# ------------------------------------------------------------
# Verify numeric values
# ------------------------------------------------------------

for predictor in static_predictors:

    if not np.isfinite(
        static_df[predictor].to_numpy(
            dtype=float
        )
    ).all():

        raise ValueError(
            f"Invalid values detected in "
            f"static predictor: {predictor}"
        )


# ------------------------------------------------------------
# Pearson correlation
# ------------------------------------------------------------

static_correlation = (
    static_df
    .corr(
        method="pearson"
    )
)


# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

static_correlation_path = (
    correlation_table_dir
    / "static_predictor_pearson_correlation.csv"
)

static_correlation.to_csv(
    static_correlation_path
)


print(
    static_correlation.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# Save heatmap
# ------------------------------------------------------------

static_heatmap_path = (
    correlation_figure_dir
    / "static_predictor_pearson_correlation.png"
)

save_correlation_heatmap(
    static_correlation,
    "Static Predictors — Pearson Correlation",
    static_heatmap_path
)

print(
    f"✓ Static correlation saved: "
    f"{static_correlation_path}"
)

print(
    f"✓ Static heatmap saved: "
    f"{static_heatmap_path}"
)


# ============================================================
# 2. YEAR-SPECIFIC CORRELATION INCLUDING RAINFALL
# ============================================================

print("\n" + "=" * 75)
print("YEAR-SPECIFIC CORRELATION INCLUDING RAINFALL")
print("=" * 75)


yearly_correlation_matrices = {}


for year, df in predictor_tables.items():

    print("\n" + "-" * 75)
    print(f"YEAR — {year}")
    print("-" * 75)


    year_predictors = (
        static_predictors
        + [rainfall_predictor]
    )


    correlation_df = df[
        year_predictors
    ].copy()


    # --------------------------------------------------------
    # Numeric validation
    # --------------------------------------------------------

    for predictor in year_predictors:

        if not np.isfinite(
            correlation_df[predictor]
            .to_numpy(
                dtype=float
            )
        ).all():

            raise ValueError(
                f"{year}: invalid values detected "
                f"in {predictor}."
            )


    # --------------------------------------------------------
    # Pearson correlation
    # --------------------------------------------------------

    correlation_matrix = (
        correlation_df
        .corr(
            method="pearson"
        )
    )


    yearly_correlation_matrices[
        year
    ] = correlation_matrix


    print(
        correlation_matrix.to_string(
            float_format=lambda x: f"{x:.4f}"
        )
    )


    # --------------------------------------------------------
    # Save CSV
    # --------------------------------------------------------

    csv_path = (
        correlation_table_dir
        / f"pearson_correlation_{year}.csv"
    )

    correlation_matrix.to_csv(
        csv_path
    )


    # --------------------------------------------------------
    # Save heatmap
    # --------------------------------------------------------

    figure_path = (
        correlation_figure_dir
        / f"pearson_correlation_{year}.png"
    )

    save_correlation_heatmap(
        correlation_matrix,
        f"Continuous Predictors + Rainfall — {year}",
        figure_path
    )


    print(
        f"✓ CSV saved    : {csv_path}"
    )

    print(
        f"✓ Heatmap saved: {figure_path}"
    )


# ============================================================
# 3. BASIC CORRELATION QA
# ============================================================

print("\n" + "=" * 75)
print("CORRELATION MATRIX QA")
print("=" * 75)


all_matrices = {
    "static": static_correlation,
    **{
        str(year): matrix
        for year, matrix
        in yearly_correlation_matrices.items()
    }
}


for name, matrix in all_matrices.items():

    # --------------------------------------------------------
    # Square matrix
    # --------------------------------------------------------

    if (
        matrix.shape[0]
        != matrix.shape[1]
    ):

        raise ValueError(
            f"{name}: correlation matrix "
            "is not square."
        )


    # --------------------------------------------------------
    # Symmetry
    # --------------------------------------------------------

    if not np.allclose(
        matrix.values,
        matrix.values.T,
        atol=1e-10
    ):

        raise ValueError(
            f"{name}: correlation matrix "
            "is not symmetric."
        )


    # --------------------------------------------------------
    # Diagonal
    # --------------------------------------------------------

    if not np.allclose(
        np.diag(matrix.values),
        1.0,
        atol=1e-10
    ):

        raise ValueError(
            f"{name}: diagonal correlation "
            "values are not equal to 1."
        )


    # --------------------------------------------------------
    # Range
    # --------------------------------------------------------

    if (
        (matrix.values < -1.0)
        |
        (matrix.values > 1.0)
    ).any():

        raise ValueError(
            f"{name}: correlation values "
            "outside [-1, 1] detected."
        )


    print(
        f"✓ {name}: matrix structure valid"
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.6 — FINAL STATUS")
print("=" * 75)

print(
    "✓ Static continuous predictors analysed"
)

print(
    "✓ Pearson correlation coefficients calculated"
)

print(
    "✓ Year-specific rainfall correlations calculated"
)

print(
    "✓ LULC excluded from Pearson correlation"
)

print(
    "✓ Correlation matrices passed structural QA"
)

print(
    "✓ Static correlation CSV and heatmap saved"
)

print(
    "✓ 2003 correlation CSV and heatmap saved"
)

print(
    "✓ 2014 correlation CSV and heatmap saved"
)

print(
    "✓ 2025 correlation CSV and heatmap saved"
)

print(
    "✓ No predictor values were modified"
)

print(
    "✓ No predictor was removed"
)

print("=" * 75)
print("✓ STEP 07.6 COMPLETED")
print("=" * 75)


NOTEBOOK 07 — CONTINUOUS PREDICTOR CORRELATION ANALYSIS

---------------------------------------------------------------------------
STATIC PREDICTOR CORRELATION
---------------------------------------------------------------------------
                   Elevation   Slope  Flow_Accumulation  River_Distance  Drainage_Density    Clay    Sand
Elevation             1.0000  0.2513            -0.2645          0.2990           -0.3003  0.1577 -0.0904
Slope                 0.2513  1.0000             0.1544          0.0121            0.0209 -0.0756 -0.1636
Flow_Accumulation    -0.2645  0.1544             1.0000         -0.1527            0.2079 -0.0050  0.0303
River_Distance        0.2990  0.0121            -0.1527          1.0000           -0.5720  0.0400 -0.0556
Drainage_Density     -0.3003  0.0209             0.2079         -0.5720            1.0000  0.0355  0.1035
Clay                  0.1577 -0.0756            -0.0050          0.0400            0.0355  1.0000  0.8936
Sand               

## 7.7 — Multicollinearity Assessment Using Variance Inflation Factor

Variance Inflation Factor (VIF) analysis is performed to assess
multicollinearity among the continuous environmental predictors.

Unlike pairwise Pearson correlation, VIF evaluates the extent to which a
predictor can be explained by the remaining predictors collectively.

The analysis includes:

- Elevation
- Slope
- Flow Accumulation
- River Distance
- Drainage Density
- Clay
- Sand
- Monsoon Rainfall

LULC is excluded because it is a categorical predictor.

Two VIF assessments are performed:

1. A static-predictor assessment using the seven static continuous
   predictors.
2. Year-specific assessments using the seven static predictors together
   with rainfall for 2003, 2014, and 2025.

VIF is interpreted as a diagnostic measure of predictor redundancy. High VIF
values indicate that a predictor contains information that overlaps strongly
with other predictors.

The following values are used as general diagnostic guidance:

- VIF close to 1: little evidence of multicollinearity
- VIF between 1 and 5: generally acceptable
- VIF between 5 and 10: potentially important multicollinearity
- VIF greater than 10: strong multicollinearity concern

These thresholds are used for interpretation only. No predictor is removed
automatically on the basis of VIF alone.

The VIF results are used together with the Pearson correlation analysis and
the physical meaning of each predictor when making later modelling
decisions.

No predictor values are modified, transformed, or removed during this step.

In [7]:
# ============================================================
# Step 07.7 — Multicollinearity Assessment Using VIF
# ============================================================

import numpy as np
import pandas as pd


print("\n" + "=" * 75)
print("NOTEBOOK 07 — MULTICOLLINEARITY ASSESSMENT")
print("=" * 75)


# ============================================================
# PREDICTOR DEFINITIONS
# ============================================================

static_predictors = [
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand"
]

rainfall_predictor = "Rainfall"


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

vif_table_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
    / "vif"
)

vif_table_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# VIF CALCULATION FUNCTION
# ============================================================

def calculate_vif(dataframe, predictors):

    X = dataframe[predictors].copy()

    # --------------------------------------------------------
    # Convert to numeric
    # --------------------------------------------------------

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )


    # --------------------------------------------------------
    # Validate missing values
    # --------------------------------------------------------

    if X.isna().any().any():

        raise ValueError(
            "Missing or non-numeric values detected "
            "before VIF calculation."
        )


    # --------------------------------------------------------
    # Validate infinite values
    # --------------------------------------------------------

    X_array = X.to_numpy(
        dtype=float
    )

    if not np.isfinite(
        X_array
    ).all():

        raise ValueError(
            "Infinite values detected "
            "before VIF calculation."
        )


    # --------------------------------------------------------
    # Standardise predictors internally
    #
    # This does NOT modify the source dataframe.
    # It only improves numerical stability because predictors
    # have very different units and magnitudes.
    # --------------------------------------------------------

    means = X_array.mean(
        axis=0
    )

    standard_deviations = X_array.std(
        axis=0,
        ddof=0
    )


    if (
        standard_deviations == 0
    ).any():

        zero_variance_predictors = [
            predictors[i]
            for i, value
            in enumerate(
                standard_deviations
            )
            if value == 0
        ]

        raise ValueError(
            "Zero-variance predictor(s) detected: "
            f"{zero_variance_predictors}"
        )


    X_standardized = (
        X_array - means
    ) / standard_deviations


    # --------------------------------------------------------
    # Calculate VIF for each predictor
    # --------------------------------------------------------

    vif_records = []


    for i, predictor in enumerate(
        predictors
    ):

        # ----------------------------------------------------
        # Target predictor
        # ----------------------------------------------------

        y = X_standardized[
            :, i
        ]


        # ----------------------------------------------------
        # Remaining predictors
        # ----------------------------------------------------

        other_indices = [
            j
            for j in range(
                len(predictors)
            )
            if j != i
        ]


        X_other = X_standardized[
            :,
            other_indices
        ]


        # ----------------------------------------------------
        # Add intercept
        # ----------------------------------------------------

        X_design = np.column_stack(
            [
                np.ones(
                    X_other.shape[0]
                ),
                X_other
            ]
        )


        # ----------------------------------------------------
        # Least-squares regression
        # ----------------------------------------------------

        coefficients = np.linalg.lstsq(
            X_design,
            y,
            rcond=None
        )[0]


        # ----------------------------------------------------
        # Predicted values
        # ----------------------------------------------------

        y_predicted = (
            X_design
            @ coefficients
        )


        # ----------------------------------------------------
        # Calculate R²
        # ----------------------------------------------------

        residual_sum_squares = np.sum(
            (
                y
                - y_predicted
            ) ** 2
        )

        total_sum_squares = np.sum(
            (
                y
                - y.mean()
            ) ** 2
        )


        if total_sum_squares == 0:

            raise ValueError(
                f"{predictor}: zero total "
                "variance detected."
            )


        r_squared = (
            1
            - (
                residual_sum_squares
                / total_sum_squares
            )
        )


        # ----------------------------------------------------
        # Numerical protection
        # ----------------------------------------------------

        r_squared = float(
            np.clip(
                r_squared,
                0.0,
                1.0 - 1e-12
            )
        )


        # ----------------------------------------------------
        # VIF
        # ----------------------------------------------------

        vif = (
            1.0
            / (
                1.0
                - r_squared
            )
        )


        vif_records.append({

            "predictor": predictor,

            "R_squared": r_squared,

            "VIF": float(vif)
        })


    return pd.DataFrame(
        vif_records
    )


# ============================================================
# VIF INTERPRETATION
# ============================================================

def classify_vif(vif):

    if vif < 5:

        return "Acceptable"

    elif vif < 10:

        return "Potential concern"

    else:

        return "Strong concern"


# ============================================================
# 1. STATIC PREDICTOR VIF
# ============================================================

print("\n" + "-" * 75)
print("STATIC PREDICTOR VIF")
print("-" * 75)


static_df = predictor_tables[2003][
    static_predictors
].copy()


static_vif = calculate_vif(
    static_df,
    static_predictors
)


static_vif[
    "interpretation"
] = static_vif[
    "VIF"
].apply(
    classify_vif
)


print(
    static_vif.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# Save static VIF
# ------------------------------------------------------------

static_vif_path = (
    vif_table_dir
    / "static_predictor_vif.csv"
)

static_vif.to_csv(
    static_vif_path,
    index=False
)


print(
    f"\n✓ Static VIF saved: "
    f"{static_vif_path}"
)


# ============================================================
# 2. YEAR-SPECIFIC VIF INCLUDING RAINFALL
# ============================================================

print("\n" + "=" * 75)
print("YEAR-SPECIFIC VIF INCLUDING RAINFALL")
print("=" * 75)


yearly_vif = {}


for year, df in predictor_tables.items():

    print("\n" + "-" * 75)
    print(f"YEAR — {year}")
    print("-" * 75)


    year_predictors = (
        static_predictors
        + [rainfall_predictor]
    )


    year_df = df[
        year_predictors
    ].copy()


    vif_result = calculate_vif(
        year_df,
        year_predictors
    )


    vif_result[
        "interpretation"
    ] = vif_result[
        "VIF"
    ].apply(
        classify_vif
    )


    yearly_vif[
        year
    ] = vif_result


    print(
        vif_result.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )


    # --------------------------------------------------------
    # Save year-specific VIF
    # --------------------------------------------------------

    vif_path = (
        vif_table_dir
        / f"predictor_vif_{year}.csv"
    )

    vif_result.to_csv(
        vif_path,
        index=False
    )


    print(
        f"\n✓ VIF saved: "
        f"{vif_path}"
    )


# ============================================================
# 3. VIF QA
# ============================================================

print("\n" + "=" * 75)
print("VIF QA")
print("=" * 75)


all_vif_tables = {

    "static": static_vif,

    **{
        str(year): table
        for year, table
        in yearly_vif.items()
    }
}


for name, table in all_vif_tables.items():

    # --------------------------------------------------------
    # Expected predictor count
    # --------------------------------------------------------

    if name == "static":

        expected_count = len(
            static_predictors
        )

    else:

        expected_count = (
            len(static_predictors)
            + 1
        )


    if len(table) != expected_count:

        raise ValueError(
            f"{name}: unexpected number "
            "of VIF records."
        )


    # --------------------------------------------------------
    # Check VIF values
    # --------------------------------------------------------

    if not np.isfinite(
        table["VIF"].to_numpy(
            dtype=float
        )
    ).all():

        raise ValueError(
            f"{name}: invalid VIF values detected."
        )


    # --------------------------------------------------------
    # VIF lower bound
    # --------------------------------------------------------

    if (
        table["VIF"] < 1.0 - 1e-10
    ).any():

        raise ValueError(
            f"{name}: VIF value below 1 detected."
        )


    print(
        f"✓ {name}: VIF structure valid"
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.7 — FINAL STATUS")
print("=" * 75)

print(
    "✓ Static predictor VIF calculated"
)

print(
    "✓ 2003 predictor VIF calculated"
)

print(
    "✓ 2014 predictor VIF calculated"
)

print(
    "✓ 2025 predictor VIF calculated"
)

print(
    "✓ VIF values passed numerical QA"
)

print(
    "✓ LULC excluded from VIF analysis"
)

print(
    "✓ No predictor was removed"
)

print(
    "✓ No predictor values were modified"
)

print(
    "✓ VIF results saved as CSV tables"
)

print("=" * 75)
print("✓ STEP 07.7 COMPLETED")
print("=" * 75)


NOTEBOOK 07 — MULTICOLLINEARITY ASSESSMENT

---------------------------------------------------------------------------
STATIC PREDICTOR VIF
---------------------------------------------------------------------------
        predictor  R_squared    VIF    interpretation
        Elevation     0.4220 1.7300        Acceptable
            Slope     0.1431 1.1670        Acceptable
Flow_Accumulation     0.1402 1.1631        Acceptable
   River_Distance     0.3500 1.5384        Acceptable
 Drainage_Density     0.3635 1.5712        Acceptable
             Clay     0.8581 7.0450 Potential concern
             Sand     0.8559 6.9388 Potential concern

✓ Static VIF saved: d:\Projects\GeoAI-Flood-Susceptibility\outputs\tables\notebook_07\vif\static_predictor_vif.csv

YEAR-SPECIFIC VIF INCLUDING RAINFALL

---------------------------------------------------------------------------
YEAR — 2003
---------------------------------------------------------------------------
        predictor  R_squared   

## 7.8 — Predictor Relationship and Multicollinearity Summary

The Pearson correlation and Variance Inflation Factor (VIF) results are
combined to provide an overall diagnostic assessment of the continuous
predictors.

For each continuous predictor, the analysis identifies:

- Maximum absolute pairwise Pearson correlation with another predictor
- Predictor corresponding to the maximum absolute correlation
- VIF value
- VIF diagnostic category

The purpose of this step is to identify potentially redundant predictors
before flood-susceptibility modelling.

Predictors are not automatically removed based on correlation or VIF
thresholds. Statistical diagnostics are considered together with the physical
meaning of the predictors and the characteristics of the modelling
algorithms.

The current analysis particularly evaluates the relationship between Clay
and Sand, which showed a strong positive Pearson correlation and elevated VIF
values.

The results are retained as a diagnostic record for later feature-selection
and model-development decisions.

No predictor values are modified and no predictor is removed during this
step.

In [8]:
# ============================================================
# Step 07.8 — Predictor Relationship and Multicollinearity
#             Summary
# ============================================================

import numpy as np
import pandas as pd


print("\n" + "=" * 75)
print("NOTEBOOK 07 — PREDICTOR RELATIONSHIP SUMMARY")
print("=" * 75)


# ============================================================
# STATIC PREDICTORS
# ============================================================

static_predictors = [
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand"
]


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

summary_table_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
    / "diagnostics"
)

summary_table_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD STATIC CORRELATION MATRIX
# ============================================================

static_correlation_path = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
    / "correlation"
    / "static_predictor_pearson_correlation.csv"
)


if not static_correlation_path.exists():

    raise FileNotFoundError(
        "Static Pearson correlation matrix not found:\n"
        f"{static_correlation_path}"
    )


static_correlation = pd.read_csv(
    static_correlation_path,
    index_col=0
)


# ============================================================
# LOAD STATIC VIF
# ============================================================

static_vif_path = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
    / "vif"
    / "static_predictor_vif.csv"
)


if not static_vif_path.exists():

    raise FileNotFoundError(
        "Static VIF table not found:\n"
        f"{static_vif_path}"
    )


static_vif = pd.read_csv(
    static_vif_path
)


# ============================================================
# VERIFY INPUTS
# ============================================================

if set(static_predictors) != set(
    static_correlation.index
):

    raise ValueError(
        "Static correlation matrix does not "
        "contain the expected predictors."
    )


if set(static_predictors) != set(
    static_vif["predictor"]
):

    raise ValueError(
        "Static VIF table does not contain "
        "the expected predictors."
    )


# ============================================================
# CREATE SUMMARY
# ============================================================

summary_records = []


for predictor in static_predictors:

    # --------------------------------------------------------
    # Correlations with all other predictors
    # --------------------------------------------------------

    correlations = (
        static_correlation.loc[
            predictor
        ]
        .drop(
            labels=[predictor]
        )
        .astype(float)
    )


    # --------------------------------------------------------
    # Maximum absolute correlation
    # --------------------------------------------------------

    strongest_predictor = (
        correlations.abs()
        .idxmax()
    )

    strongest_correlation = (
        correlations[
            strongest_predictor
        ]
    )

    maximum_absolute_correlation = abs(
        strongest_correlation
    )


    # --------------------------------------------------------
    # VIF
    # --------------------------------------------------------

    vif_row = static_vif[
        static_vif["predictor"]
        == predictor
    ]


    if len(vif_row) != 1:

        raise ValueError(
            f"Expected exactly one VIF "
            f"record for {predictor}."
        )


    vif_value = float(
        vif_row.iloc[0]["VIF"]
    )


    vif_interpretation = (
        str(
            vif_row.iloc[0][
                "interpretation"
            ]
        )
    )


    # --------------------------------------------------------
    # Diagnostic flag
    # --------------------------------------------------------

    if (
        maximum_absolute_correlation >= 0.80
        or
        vif_value >= 5.0
    ):

        diagnostic_flag = (
            "Further investigation"
        )

    else:

        diagnostic_flag = (
            "No major concern"
        )


    summary_records.append({

        "predictor":
            predictor,

        "strongest_correlated_predictor":
            strongest_predictor,

        "pearson_r":
            float(
                strongest_correlation
            ),

        "absolute_pearson_r":
            float(
                maximum_absolute_correlation
            ),

        "VIF":
            vif_value,

        "VIF_interpretation":
            vif_interpretation,

        "diagnostic_flag":
            diagnostic_flag
    })


# ============================================================
# CREATE DATAFRAME
# ============================================================

predictor_relationship_summary = pd.DataFrame(
    summary_records
)


# ============================================================
# SORT BY DIAGNOSTIC SEVERITY
# ============================================================

predictor_relationship_summary = (
    predictor_relationship_summary
    .sort_values(
        by=[
            "VIF",
            "absolute_pearson_r"
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# SAVE SUMMARY
# ============================================================

summary_path = (
    summary_table_dir
    / "predictor_relationship_multicollinearity_summary.csv"
)

predictor_relationship_summary.to_csv(
    summary_path,
    index=False
)


# ============================================================
# DISPLAY SUMMARY
# ============================================================

print("\n" + "-" * 75)
print("PREDICTOR RELATIONSHIP SUMMARY")
print("-" * 75)

print(
    predictor_relationship_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# IDENTIFY FLAGGED PREDICTORS
# ============================================================

flagged_predictors = (
    predictor_relationship_summary[
        predictor_relationship_summary[
            "diagnostic_flag"
        ]
        == "Further investigation"
    ]["predictor"]
    .tolist()
)


print("\n" + "-" * 75)
print("PREDICTORS FLAGGED FOR FURTHER INVESTIGATION")
print("-" * 75)


if flagged_predictors:

    for predictor in flagged_predictors:

        print(
            f"⚠ {predictor}"
        )

else:

    print(
        "✓ No predictors flagged."
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.8 — FINAL STATUS")
print("=" * 75)

print(
    "✓ Pearson correlation diagnostics loaded"
)

print(
    "✓ VIF diagnostics loaded"
)

print(
    "✓ Maximum pairwise correlations identified"
)

print(
    "✓ VIF values incorporated"
)

print(
    "✓ Potentially redundant predictors flagged"
)

print(
    f"✓ Summary saved: {summary_path}"
)

print(
    "✓ No predictor was removed"
)

print(
    "✓ No predictor values were modified"
)

print("=" * 75)
print("✓ STEP 07.8 COMPLETED")
print("=" * 75)


NOTEBOOK 07 — PREDICTOR RELATIONSHIP SUMMARY

---------------------------------------------------------------------------
PREDICTOR RELATIONSHIP SUMMARY
---------------------------------------------------------------------------
        predictor strongest_correlated_predictor  pearson_r  absolute_pearson_r    VIF VIF_interpretation       diagnostic_flag
             Clay                           Sand     0.8936              0.8936 7.0450  Potential concern Further investigation
             Sand                           Clay     0.8936              0.8936 6.9388  Potential concern Further investigation
        Elevation               Drainage_Density    -0.3003              0.3003 1.7300         Acceptable      No major concern
 Drainage_Density                 River_Distance    -0.5720              0.5720 1.5712         Acceptable      No major concern
   River_Distance               Drainage_Density    -0.5720              0.5720 1.5384         Acceptable      No major concern
  

## 7.9 — Temporal Predictor Consistency Assessment

The three year-specific predictor tables contain both static and
time-varying predictors.

The purpose of this step is to distinguish predictors that are expected to
remain spatially constant from predictors that legitimately change between
2003, 2014, and 2025.

The static continuous predictors are:

- Elevation
- Slope
- Flow Accumulation
- River Distance
- Drainage Density
- Clay
- Sand

These predictors are derived from static or time-invariant source datasets
and are therefore expected to remain unchanged across the three modelling
years.

The temporal predictor considered in this step is:

- Monsoon Rainfall

Rainfall is expected to vary between years and is therefore assessed for
temporal change.

LULC is not re-analysed in this step because its temporal composition and
area changes were already assessed in Step 07.4.

For the static predictors, year-to-year consistency is evaluated using:

- Number of observations
- Mean
- Standard deviation
- Minimum
- Maximum
- Absolute difference in yearly means

For rainfall, the year-specific statistics are compared to quantify its
temporal variability.

The purpose is to verify that static predictors have remained consistent
throughout the integrated datasets while confirming that rainfall retains
its expected temporal variability.

No predictor values are modified, transformed, or removed during this step.

In [9]:
# ============================================================
# Step 07.9 — Temporal Predictor Consistency Assessment
# ============================================================

import numpy as np
import pandas as pd


print("\n" + "=" * 75)
print("NOTEBOOK 07 — TEMPORAL PREDICTOR CONSISTENCY ASSESSMENT")
print("=" * 75)


# ============================================================
# PREDICTOR DEFINITIONS
# ============================================================

static_predictors = [
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand"
]

rainfall_predictor = "Rainfall"

years = [
    2003,
    2014,
    2025
]


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

temporal_output_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
    / "temporal_consistency"
)

temporal_output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# VERIFY PREDICTOR TABLES
# ============================================================

for year in years:

    if year not in predictor_tables:

        raise ValueError(
            f"Predictor table for {year} "
            "is not available."
        )


# ============================================================
# 1. STATIC PREDICTOR YEAR-WISE STATISTICS
# ============================================================

print("\n" + "-" * 75)
print("STATIC PREDICTOR YEAR-WISE CONSISTENCY")
print("-" * 75)


static_records = []


for year in years:

    df = predictor_tables[year]


    for predictor in static_predictors:

        values = pd.to_numeric(
            df[predictor],
            errors="coerce"
        )


        if values.isna().any():

            raise ValueError(
                f"{predictor} contains missing "
                f"values in {year}."
            )


        if not np.isfinite(
            values.to_numpy(
                dtype=float
            )
        ).all():

            raise ValueError(
                f"{predictor} contains infinite "
                f"values in {year}."
            )


        static_records.append({

            "year":
                year,

            "predictor":
                predictor,

            "count":
                int(values.count()),

            "mean":
                float(values.mean()),

            "std":
                float(values.std()),

            "minimum":
                float(values.min()),

            "maximum":
                float(values.max())
        })


static_statistics = pd.DataFrame(
    static_records
)


print(
    static_statistics.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# SAVE STATIC STATISTICS
# ============================================================

static_statistics_path = (
    temporal_output_dir
    / "static_predictor_temporal_consistency.csv"
)

static_statistics.to_csv(
    static_statistics_path,
    index=False
)


print(
    f"\n✓ Static consistency table saved: "
    f"{static_statistics_path}"
)


# ============================================================
# 2. STATIC PREDICTOR MEAN DIFFERENCE
# ============================================================

print("\n" + "-" * 75)
print("STATIC PREDICTOR MEAN CONSISTENCY")
print("-" * 75)


static_mean_records = []


for predictor in static_predictors:

    yearly_means = {}


    for year in years:

        row = static_statistics[
            (
                static_statistics["year"]
                == year
            )
            &
            (
                static_statistics["predictor"]
                == predictor
            )
        ]


        if len(row) != 1:

            raise ValueError(
                f"Expected one statistics "
                f"record for {predictor}, "
                f"{year}."
            )


        yearly_means[year] = float(
            row.iloc[0]["mean"]
        )


    mean_2003 = yearly_means[2003]
    mean_2014 = yearly_means[2014]
    mean_2025 = yearly_means[2025]


    max_mean_difference = (
        max(
            yearly_means.values()
        )
        -
        min(
            yearly_means.values()
        )
    )


    static_mean_records.append({

        "predictor":
            predictor,

        "mean_2003":
            mean_2003,

        "mean_2014":
            mean_2014,

        "mean_2025":
            mean_2025,

        "maximum_absolute_mean_difference":
            float(max_mean_difference)
    })


static_mean_comparison = pd.DataFrame(
    static_mean_records
)


print(
    static_mean_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# SAVE STATIC MEAN COMPARISON
# ============================================================

static_mean_path = (
    temporal_output_dir
    / "static_predictor_mean_consistency.csv"
)

static_mean_comparison.to_csv(
    static_mean_path,
    index=False
)


print(
    f"\n✓ Static mean comparison saved: "
    f"{static_mean_path}"
)


# ============================================================
# 3. STATIC PREDICTOR EXACT VALUE CONSISTENCY
# ============================================================

print("\n" + "-" * 75)
print("STATIC PREDICTOR VALUE CONSISTENCY")
print("-" * 75)


static_exact_records = []


for predictor in static_predictors:

    reference_df = predictor_tables[
        2003
    ]

    reference_values = (
        reference_df[
            predictor
        ]
        .to_numpy(
            dtype=float
        )
    )


    all_identical = True


    for year in [
        2014,
        2025
    ]:

        current_df = predictor_tables[
            year
        ]


        # ----------------------------------------------------
        # Static predictor tables can have different row
        # counts because the valid-cell masks differ.
        #
        # Therefore exact array comparison is not appropriate.
        # Instead, compare values using row/column spatial
        # coordinates where the same cells exist.
        # ----------------------------------------------------

        common_columns = [
            "row",
            "col",
            predictor
        ]


        reference_subset = reference_df[
            common_columns
        ].copy()


        current_subset = current_df[
            common_columns
        ].copy()


        merged = reference_subset.merge(
            current_subset,
            on=[
                "row",
                "col"
            ],
            suffixes=(
                "_reference",
                "_current"
            )
        )


        if len(merged) == 0:

            raise ValueError(
                f"No common spatial cells "
                f"found for {predictor} "
                f"between 2003 and {year}."
            )


        differences = (
            merged[
                f"{predictor}_reference"
            ].to_numpy(
                dtype=float
            )
            -
            merged[
                f"{predictor}_current"
            ].to_numpy(
                dtype=float
            )
        )


        maximum_difference = np.max(
            np.abs(
                differences
            )
        )


        if not np.isclose(
            maximum_difference,
            0.0,
            atol=1e-6
        ):

            all_identical = False


    static_exact_records.append({

        "predictor":
            predictor,

        "static_values_consistent":
            all_identical
    })


static_exact_consistency = pd.DataFrame(
    static_exact_records
)


print(
    static_exact_consistency.to_string(
        index=False
    )
)


# ============================================================
# SAVE EXACT CONSISTENCY RESULT
# ============================================================

static_exact_path = (
    temporal_output_dir
    / "static_predictor_value_consistency.csv"
)

static_exact_consistency.to_csv(
    static_exact_path,
    index=False
)


print(
    f"\n✓ Static value consistency saved: "
    f"{static_exact_path}"
)


# ============================================================
# 4. RAINFALL TEMPORAL STATISTICS
# ============================================================

print("\n" + "=" * 75)
print("RAINFALL TEMPORAL VARIABILITY")
print("=" * 75)


rainfall_records = []


for year in years:

    df = predictor_tables[
        year
    ]


    values = pd.to_numeric(
        df[rainfall_predictor],
        errors="coerce"
    )


    if values.isna().any():

        raise ValueError(
            f"Rainfall contains missing "
            f"values in {year}."
        )


    if not np.isfinite(
        values.to_numpy(
            dtype=float
        )
    ).all():

        raise ValueError(
            f"Rainfall contains infinite "
            f"values in {year}."
        )


    rainfall_records.append({

        "year":
            year,

        "count":
            int(values.count()),

        "mean_mm":
            float(values.mean()),

        "std_mm":
            float(values.std()),

        "minimum_mm":
            float(values.min()),

        "median_mm":
            float(values.median()),

        "maximum_mm":
            float(values.max())
    })


rainfall_temporal_statistics = pd.DataFrame(
    rainfall_records
)


print(
    rainfall_temporal_statistics.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# SAVE RAINFALL TEMPORAL STATISTICS
# ============================================================

rainfall_statistics_path = (
    temporal_output_dir
    / "rainfall_temporal_consistency.csv"
)

rainfall_temporal_statistics.to_csv(
    rainfall_statistics_path,
    index=False
)


print(
    f"\n✓ Rainfall temporal table saved: "
    f"{rainfall_statistics_path}"
)


# ============================================================
# 5. RAINFALL MEAN CHANGES
# ============================================================

print("\n" + "-" * 75)
print("RAINFALL MEAN TEMPORAL CHANGE")
print("-" * 75)


rainfall_means = (
    rainfall_temporal_statistics
    .set_index("year")[
        "mean_mm"
    ]
)


rainfall_change_records = []


comparisons = [
    (2003, 2014),
    (2014, 2025),
    (2003, 2025)
]


for initial_year, final_year in comparisons:

    initial_mean = float(
        rainfall_means[
            initial_year
        ]
    )

    final_mean = float(
        rainfall_means[
            final_year
        ]
    )


    absolute_change = (
        final_mean
        -
        initial_mean
    )


    percentage_change = (
        (
            absolute_change
            /
            initial_mean
        )
        * 100
    )


    rainfall_change_records.append({

        "comparison":
            f"{initial_year}_to_{final_year}",

        "initial_year":
            initial_year,

        "final_year":
            final_year,

        "initial_mean_mm":
            initial_mean,

        "final_mean_mm":
            final_mean,

        "absolute_change_mm":
            absolute_change,

        "percentage_change":
            percentage_change
    })


rainfall_change = pd.DataFrame(
    rainfall_change_records
)


print(
    rainfall_change.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# SAVE RAINFALL CHANGE TABLE
# ============================================================

rainfall_change_path = (
    temporal_output_dir
    / "rainfall_temporal_mean_change.csv"
)

rainfall_change.to_csv(
    rainfall_change_path,
    index=False
)


print(
    f"\n✓ Rainfall change table saved: "
    f"{rainfall_change_path}"
)


# ============================================================
# 6. FINAL QA
# ============================================================

print("\n" + "=" * 75)
print("TEMPORAL CONSISTENCY QA")
print("=" * 75)


# ------------------------------------------------------------
# Static predictor count
# ------------------------------------------------------------

if len(
    static_statistics
) != (
    len(static_predictors)
    * len(years)
):

    raise ValueError(
        "Unexpected number of static "
        "predictor statistics."
    )


# ------------------------------------------------------------
# Rainfall year count
# ------------------------------------------------------------

if len(
    rainfall_temporal_statistics
) != len(years):

    raise ValueError(
        "Unexpected rainfall year count."
    )


# ------------------------------------------------------------
# Static value consistency
# ------------------------------------------------------------

if not static_exact_consistency[
    "static_values_consistent"
].all():

    print(
        "⚠ One or more static predictors "
        "show value differences across "
        "common spatial cells."
    )

else:

    print(
        "✓ All static predictors remain "
        "spatially consistent across years."
    )


# ------------------------------------------------------------
# Rainfall variability
# ------------------------------------------------------------

rainfall_mean_range = (
    rainfall_means.max()
    -
    rainfall_means.min()
)


if rainfall_mean_range == 0:

    print(
        "⚠ Rainfall means are identical "
        "across all years."
    )

else:

    print(
        "✓ Rainfall exhibits temporal "
        "variability across years."
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.9 — FINAL STATUS")
print("=" * 75)

print(
    "✓ Static predictor year-wise "
    "statistics calculated"
)

print(
    "✓ Static predictor mean consistency "
    "assessed"
)

print(
    "✓ Static predictor spatial-value "
    "consistency assessed"
)

print(
    "✓ Rainfall temporal statistics "
    "calculated"
)

print(
    "✓ Rainfall temporal changes "
    "calculated"
)

print(
    "✓ No predictor values were modified"
)

print(
    "✓ No predictor was removed"
)

print(
    "✓ LULC was not duplicated from Step 07.4"
)

print(
    "✓ Temporal consistency tables saved"
)

print("=" * 75)
print("✓ STEP 07.9 COMPLETED")
print("=" * 75)


NOTEBOOK 07 — TEMPORAL PREDICTOR CONSISTENCY ASSESSMENT

---------------------------------------------------------------------------
STATIC PREDICTOR YEAR-WISE CONSISTENCY
---------------------------------------------------------------------------
 year         predictor  count      mean        std  minimum     maximum
 2003         Elevation  47523  120.6940     4.8974 101.2746    135.9356
 2003             Slope  47523    1.5668     0.9564   0.0000      8.3203
 2003 Flow_Accumulation  47523 3913.0912 26416.9663   2.1022 582647.0600
 2003    River_Distance  47523 1277.0529   879.9139  38.3393   5129.2850
 2003  Drainage_Density  47523    0.3929     0.3959   0.0000      2.1059
 2003              Clay  47523   25.1674     6.4413   0.0000     33.8261
 2003              Sand  47523   32.0595     7.9556   0.0000     42.5395
 2014         Elevation  46391  120.6396     4.9387 101.2746    135.9356
 2014             Slope  46391    1.5782     0.9580   0.0000      8.3203
 2014 Flow_Accumulati

## 7.10 — Final Predictor Dataset Summary

The final year-specific predictor datasets are summarised to establish the
dataset structure available for subsequent flood-susceptibility modelling.

The integrated datasets represent three modelling years:

- 2003
- 2014
- 2025

Each dataset contains the seven static continuous predictors:

- Elevation
- Slope
- Flow Accumulation
- River Distance
- Drainage Density
- Clay
- Sand

and two year-specific predictors:

- Monsoon Rainfall
- LULC

The summary records:

- Number of modelling observations
- Predictor availability
- LULC class availability
- Missing-value status
- Infinite-value status
- Continuous predictor ranges
- Year-specific modelling coverage

The number of available observations differs between years because the
common valid-cell masks differ, particularly due to LULC and the valid
coverage of the original predictor datasets.

These differences are retained rather than filled or artificially equalised.

The final predictor datasets are considered ready for integration with the
flood-related target data in the subsequent workflow.

No predictor values are modified, imputed, or removed during this step.

In [10]:
# ============================================================
# Step 07.10 — Final Predictor Dataset Summary
# ============================================================

import numpy as np
import pandas as pd


print("\n" + "=" * 75)
print("NOTEBOOK 07 — FINAL PREDICTOR DATASET SUMMARY")
print("=" * 75)


# ============================================================
# PREDICTOR DEFINITIONS
# ============================================================

static_predictors = [
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand"
]

dynamic_predictors = [
    "Rainfall",
    "LULC"
]

continuous_predictors = (
    static_predictors
    + ["Rainfall"]
)

years = [
    2003,
    2014,
    2025
]


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

summary_output_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "notebook_07"
    / "final_summary"
)

summary_output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# EXPECTED COLUMN STRUCTURE
# ============================================================

expected_columns = [
    "year",
    "row",
    "col",
    "x_utm",
    "y_utm",
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand",
    "Rainfall",
    "LULC"
]


# ============================================================
# FINAL YEAR-WISE SUMMARY
# ============================================================

summary_records = []


for year in years:

    print("\n" + "-" * 75)
    print(f"YEAR — {year}")
    print("-" * 75)


    df = predictor_tables[
        year
    ].copy()


    # --------------------------------------------------------
    # Structural verification
    # --------------------------------------------------------

    if list(df.columns) != expected_columns:

        raise ValueError(
            f"{year}: unexpected column structure."
        )


    # --------------------------------------------------------
    # Missing values
    # --------------------------------------------------------

    missing_values = int(
        df.isna()
        .sum()
        .sum()
    )


    # --------------------------------------------------------
    # Infinite values
    # --------------------------------------------------------

    numeric_df = df.select_dtypes(
        include=[np.number]
    )


    infinite_values = int(
        np.isinf(
            numeric_df.to_numpy(
                dtype=float
            )
        )
        .sum()
    )


    # --------------------------------------------------------
    # Duplicate spatial cells
    # --------------------------------------------------------

    duplicate_cells = int(
        df.duplicated(
            subset=[
                "row",
                "col"
            ]
        )
        .sum()
    )


    # --------------------------------------------------------
    # LULC classes
    # --------------------------------------------------------

    lulc_classes = sorted(
        df["LULC"]
        .dropna()
        .unique()
        .tolist()
    )


    # --------------------------------------------------------
    # Continuous predictor ranges
    # --------------------------------------------------------

    ranges = {}


    for predictor in continuous_predictors:

        values = pd.to_numeric(
            df[predictor],
            errors="coerce"
        )


        if values.isna().any():

            raise ValueError(
                f"{year}: missing values "
                f"detected in {predictor}."
            )


        if not np.isfinite(
            values.to_numpy(
                dtype=float
            )
        ).all():

            raise ValueError(
                f"{year}: infinite values "
                f"detected in {predictor}."
            )


        ranges[
            predictor
        ] = (
            float(values.min()),
            float(values.max())
        )


    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------

    print(
        f"Observations      : "
        f"{len(df):,}"
    )

    print(
        f"Columns           : "
        f"{len(df.columns)}"
    )

    print(
        f"Missing values    : "
        f"{missing_values}"
    )

    print(
        f"Infinite values   : "
        f"{infinite_values}"
    )

    print(
        f"Duplicate cells   : "
        f"{duplicate_cells}"
    )

    print(
        f"LULC classes      : "
        f"{lulc_classes}"
    )


    print(
        "\nContinuous predictor ranges:"
    )


    for predictor in continuous_predictors:

        minimum, maximum = ranges[
            predictor
        ]

        unit = (
            " mm"
            if predictor == "Rainfall"
            else ""
        )

        print(
            f"  {predictor:<20}"
            f"{minimum:.4f} – "
            f"{maximum:.4f}{unit}"
        )


    # --------------------------------------------------------
    # Store summary
    # --------------------------------------------------------

    summary_records.append({

        "year":
            year,

        "observations":
            int(len(df)),

        "columns":
            int(len(df.columns)),

        "missing_values":
            missing_values,

        "infinite_values":
            infinite_values,

        "duplicate_spatial_cells":
            duplicate_cells,

        "LULC_class_count":
            len(lulc_classes),

        "LULC_classes":
            ",".join(
                map(
                    str,
                    lulc_classes
                )
            )
    })


# ============================================================
# CREATE FINAL SUMMARY TABLE
# ============================================================

final_summary = pd.DataFrame(
    summary_records
)


# ============================================================
# SAVE FINAL SUMMARY
# ============================================================

final_summary_path = (
    summary_output_dir
    / "final_predictor_dataset_summary.csv"
)

final_summary.to_csv(
    final_summary_path,
    index=False
)


print("\n" + "=" * 75)
print("FINAL PREDICTOR DATASET SUMMARY")
print("=" * 75)

print(
    final_summary.to_string(
        index=False
    )
)


print(
    f"\n✓ Final summary saved: "
    f"{final_summary_path}"
)


# ============================================================
# FINAL QA
# ============================================================

print("\n" + "=" * 75)
print("FINAL PREDICTOR DATASET QA")
print("=" * 75)


for _, row in final_summary.iterrows():

    year = int(
        row["year"]
    )


    if row["missing_values"] != 0:

        raise ValueError(
            f"{year}: missing values detected."
        )


    if row["infinite_values"] != 0:

        raise ValueError(
            f"{year}: infinite values detected."
        )


    if row[
        "duplicate_spatial_cells"
    ] != 0:

        raise ValueError(
            f"{year}: duplicate spatial cells detected."
        )


    if row[
        "LULC_class_count"
    ] != 5:

        raise ValueError(
            f"{year}: unexpected LULC class count."
        )


    print(
        f"✓ {year}: final predictor dataset valid"
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 07.10 — FINAL STATUS")
print("=" * 75)

print(
    "✓ 2003 predictor dataset summarised"
)

print(
    "✓ 2014 predictor dataset summarised"
)

print(
    "✓ 2025 predictor dataset summarised"
)

print(
    "✓ Static predictors confirmed"
)

print(
    "✓ Dynamic predictors confirmed"
)

print(
    "✓ LULC classes confirmed"
)

print(
    "✓ No missing values detected"
)

print(
    "✓ No infinite values detected"
)

print(
    "✓ No duplicate spatial cells detected"
)

print(
    "✓ Final predictor summary saved"
)

print(
    "✓ No predictor values were modified"
)

print(
    "✓ No predictor was removed"
)

print("=" * 75)
print("✓ STEP 07.10 COMPLETED")
print("=" * 75)


NOTEBOOK 07 — FINAL PREDICTOR DATASET SUMMARY

---------------------------------------------------------------------------
YEAR — 2003
---------------------------------------------------------------------------
Observations      : 47,523
Columns           : 14
Missing values    : 0
Infinite values   : 0
Duplicate cells   : 0
LULC classes      : [1, 2, 3, 4, 5]

Continuous predictor ranges:
  Elevation           101.2746 – 135.9356
  Slope               0.0000 – 8.3203
  Flow_Accumulation   2.1022 – 582647.0600
  River_Distance      38.3393 – 5129.2850
  Drainage_Density    0.0000 – 2.1059
  Clay                0.0000 – 33.8261
  Sand                0.0000 – 42.5395
  Rainfall            155.2107 – 1003.2170 mm

---------------------------------------------------------------------------
YEAR — 2014
---------------------------------------------------------------------------


Observations      : 46,391
Columns           : 14
Missing values    : 0
Infinite values   : 0
Duplicate cells   : 0
LULC classes      : [1, 2, 3, 4, 5]

Continuous predictor ranges:
  Elevation           101.2746 – 135.9356
  Slope               0.0000 – 8.3203
  Flow_Accumulation   2.1022 – 582647.0600
  River_Distance      38.3393 – 5129.2850
  Drainage_Density    0.0000 – 2.1059
  Clay                0.0000 – 33.8261
  Sand                0.0000 – 42.5395
  Rainfall            115.3455 – 834.5916 mm

---------------------------------------------------------------------------
YEAR — 2025
---------------------------------------------------------------------------
Observations      : 43,076
Columns           : 14
Missing values    : 0
Infinite values   : 0
Duplicate cells   : 0
LULC classes      : [1, 2, 3, 4, 5]

Continuous predictor ranges:
  Elevation           101.2746 – 135.9356
  Slope               0.0000 – 8.3203
  Flow_Accumulation   2.1022 – 582647.0600
  River_Distance      

## 7.11 — Final Output Inventory and Quality Assurance

This final step verifies the completeness of all analytical outputs generated
during Notebook 07.

The output inventory covers:

- LULC composition and temporal-change tables
- Rainfall statistical and temporal-change tables
- Pearson correlation tables
- Pearson correlation heatmaps
- VIF tables
- Predictor relationship and multicollinearity summary
- Static predictor temporal-consistency tables
- Rainfall temporal-consistency tables
- Final predictor dataset summary

The inventory verifies that all required outputs exist in their designated
directories.

This step does not modify, transform, or regenerate any predictor dataset.
It only verifies the presence of the final analytical outputs.

Notebook 07 is considered complete only when all required analytical tables
and figures are present.

In [11]:
# ============================================================
# Step 07.11 — Final Output Inventory & QA
# ============================================================

from pathlib import Path


print("\n" + "=" * 75)
print("NOTEBOOK 07 — FINAL OUTPUT INVENTORY")
print("=" * 75)


# ============================================================
# EXPECTED OUTPUTS
# ============================================================

expected_tables = [

    # --------------------------------------------------------
    # LULC
    # --------------------------------------------------------

    (
        "LULC composition",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "LULC_composition_2003_2014_2025.csv"
    ),

    (
        "LULC area change",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "LULC_area_change_2003_2014_2025.csv"
    ),

    (
        "LULC percentage change",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "LULC_percentage_change_2003_2014_2025.csv"
    ),

    # --------------------------------------------------------
    # Rainfall
    # --------------------------------------------------------

    (
        "Rainfall statistics",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "rainfall_statistics_2003_2014_2025.csv"
    ),

    (
        "Rainfall mean temporal change",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "rainfall_mean_temporal_change_2003_2014_2025.csv"
    ),

    # --------------------------------------------------------
    # Correlation
    # --------------------------------------------------------

    (
        "Static Pearson correlation",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "correlation"
        / "static_predictor_pearson_correlation.csv"
    ),

    (
        "2003 Pearson correlation",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "correlation"
        / "pearson_correlation_2003.csv"
    ),

    (
        "2014 Pearson correlation",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "correlation"
        / "pearson_correlation_2014.csv"
    ),

    (
        "2025 Pearson correlation",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "correlation"
        / "pearson_correlation_2025.csv"
    ),

    # --------------------------------------------------------
    # VIF
    # --------------------------------------------------------

    (
        "Static VIF",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "vif"
        / "static_predictor_vif.csv"
    ),

    (
        "2003 VIF",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "vif"
        / "predictor_vif_2003.csv"
    ),

    (
        "2014 VIF",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "vif"
        / "predictor_vif_2014.csv"
    ),

    (
        "2025 VIF",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "vif"
        / "predictor_vif_2025.csv"
    ),

    # --------------------------------------------------------
    # Predictor diagnostics
    # --------------------------------------------------------

    (
        "Predictor relationship summary",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "diagnostics"
        / "predictor_relationship_multicollinearity_summary.csv"
    ),

    # --------------------------------------------------------
    # Temporal consistency
    # --------------------------------------------------------

    (
        "Static predictor temporal consistency",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "temporal_consistency"
        / "static_predictor_temporal_consistency.csv"
    ),

    (
        "Static predictor mean consistency",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "temporal_consistency"
        / "static_predictor_mean_consistency.csv"
    ),

    (
        "Static predictor value consistency",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "temporal_consistency"
        / "static_predictor_value_consistency.csv"
    ),

    (
        "Rainfall temporal consistency",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "temporal_consistency"
        / "rainfall_temporal_consistency.csv"
    ),

    (
        "Rainfall temporal mean change",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "temporal_consistency"
        / "rainfall_temporal_mean_change.csv"
    ),

    # --------------------------------------------------------
    # Final predictor summary
    # --------------------------------------------------------

    (
        "Final predictor dataset summary",
        PROJECT_ROOT
        / "outputs"
        / "tables"
        / "notebook_07"
        / "final_summary"
        / "final_predictor_dataset_summary.csv"
    )
]


expected_figures = [

    # --------------------------------------------------------
    # Pearson correlation heatmaps
    # --------------------------------------------------------

    (
        "Static Pearson heatmap",
        PROJECT_ROOT
        / "outputs"
        / "figures"
        / "notebook_07"
        / "correlation"
        / "static_predictor_pearson_correlation.png"
    ),

    (
        "2003 Pearson heatmap",
        PROJECT_ROOT
        / "outputs"
        / "figures"
        / "notebook_07"
        / "correlation"
        / "pearson_correlation_2003.png"
    ),

    (
        "2014 Pearson heatmap",
        PROJECT_ROOT
        / "outputs"
        / "figures"
        / "notebook_07"
        / "correlation"
        / "pearson_correlation_2014.png"
    ),

    (
        "2025 Pearson heatmap",
        PROJECT_ROOT
        / "outputs"
        / "figures"
        / "notebook_07"
        / "correlation"
        / "pearson_correlation_2025.png"
    )
]


# ============================================================
# TABLE INVENTORY
# ============================================================

print("\n" + "-" * 75)
print("TABLE OUTPUTS")
print("-" * 75)


missing_tables = []


for label, path in expected_tables:

    if path.exists():

        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"✓ {label:<42}"
            f"{size_mb:.2f} MB"
        )

    else:

        print(
            f"✗ {label:<42}"
            "MISSING"
        )

        missing_tables.append(
            (label, path)
        )


# ============================================================
# FIGURE INVENTORY
# ============================================================

print("\n" + "-" * 75)
print("FIGURE OUTPUTS")
print("-" * 75)


missing_figures = []


for label, path in expected_figures:

    if path.exists():

        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"✓ {label:<42}"
            f"{size_mb:.2f} MB"
        )

    else:

        print(
            f"✗ {label:<42}"
            "MISSING"
        )

        missing_figures.append(
            (label, path)
        )


# ============================================================
# FINAL INVENTORY QA
# ============================================================

print("\n" + "=" * 75)
print("NOTEBOOK 07 — FINAL OUTPUT QA")
print("=" * 75)


total_expected = (
    len(expected_tables)
    +
    len(expected_figures)
)


total_missing = (
    len(missing_tables)
    +
    len(missing_figures)
)


print(
    f"Expected outputs : {total_expected}"
)

print(
    f"Missing outputs  : {total_missing}"
)


if total_missing > 0:

    print(
        "\n⚠ NOTEBOOK 07 OUTPUT INVENTORY FAILED"
    )

    print(
        "\nMissing tables:"
    )

    for label, path in missing_tables:

        print(
            f"  - {label}"
        )

        print(
            f"    {path}"
        )


    print(
        "\nMissing figures:"
    )

    for label, path in missing_figures:

        print(
            f"  - {label}"
        )

        print(
            f"    {path}"
        )


    raise FileNotFoundError(
        "One or more required Notebook 07 "
        "outputs are missing."
    )


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n✓ All required analytical tables are present"
)

print(
    "✓ All required correlation heatmaps are present"
)

print(
    "✓ LULC temporal-analysis outputs are present"
)

print(
    "✓ Rainfall temporal-analysis outputs are present"
)

print(
    "✓ Pearson correlation outputs are present"
)

print(
    "✓ VIF outputs are present"
)

print(
    "✓ Multicollinearity diagnostic output is present"
)

print(
    "✓ Temporal-consistency outputs are present"
)

print(
    "✓ Final predictor summary is present"
)

print(
    "✓ No predictor dataset was modified"
)

print("\n" + "=" * 75)
print("NOTEBOOK 07 — FINAL OUTPUT STATUS")
print("=" * 75)

print(
    "✓ ALL REQUIRED NOTEBOOK 07 OUTPUTS ARE PRESENT"
)

print(
    "✓ NOTEBOOK 07 OUTPUT INVENTORY PASSED"
)

print("=" * 75)
print("✓ STEP 07.11 COMPLETED")
print("=" * 75)


NOTEBOOK 07 — FINAL OUTPUT INVENTORY

---------------------------------------------------------------------------
TABLE OUTPUTS
---------------------------------------------------------------------------
✓ LULC composition                          0.00 MB
✓ LULC area change                          0.00 MB
✓ LULC percentage change                    0.00 MB
✓ Rainfall statistics                       0.00 MB
✓ Rainfall mean temporal change             0.00 MB
✓ Static Pearson correlation                0.00 MB
✓ 2003 Pearson correlation                  0.00 MB
✓ 2014 Pearson correlation                  0.00 MB
✓ 2025 Pearson correlation                  0.00 MB
✓ Static VIF                                0.00 MB
✓ 2003 VIF                                  0.00 MB
✓ 2014 VIF                                  0.00 MB
✓ 2025 VIF                                  0.00 MB
✓ Predictor relationship summary            0.00 MB
✓ Static predictor temporal consistency     0.00 MB
✓ Static predic

# Notebook 07 — Final Summary

Notebook 07 completed the temporal and statistical analysis of the
year-specific flood-susceptibility predictor datasets for 2003, 2014, and
2025.

The notebook began with structural verification of the three integrated
predictor tables and subsequently analysed LULC composition, rainfall
variability, predictor relationships, multicollinearity, and temporal
consistency.

### Major analyses completed

1. **Predictor table verification**
   - Verified the three year-specific predictor tables.
   - Confirmed expected dimensions and column structure.
   - Confirmed year identifiers.
   - Confirmed absence of missing and infinite values.
   - Confirmed absence of duplicate spatial cells.

2. **LULC composition and temporal change**
   - Analysed Water, Vegetation, Built-up, Barren, and Agriculture classes.
   - Calculated class-wise cell counts, percentages, and approximate areas.
   - Quantified area and percentage-share changes between 2003, 2014, and
     2025.

3. **Monsoon rainfall temporal analysis**
   - Calculated descriptive statistics for 2003, 2014, and 2025.
   - Quantified absolute and percentage changes in mean rainfall.
   - Confirmed rainfall as a time-varying continuous predictor.

4. **Pearson correlation analysis**
   - Calculated correlations among the seven static continuous predictors.
   - Calculated year-specific correlations including rainfall.
   - LULC was excluded because it is categorical.
   - Correlation matrices and heatmaps were generated.

5. **Multicollinearity assessment**
   - Calculated VIF for static predictors.
   - Calculated year-specific VIF including rainfall.
   - Clay and Sand were identified as the principal multicollinearity concern.
   - No predictor was automatically removed.

6. **Predictor relationship assessment**
   - Combined Pearson correlation and VIF diagnostics.
   - Identified predictors requiring further investigation.
   - Clay and Sand were flagged because of their strong relationship.

7. **Temporal predictor consistency**
   - Verified that static predictors remain spatially consistent across
     modelling years.
   - Confirmed that differences in year-specific summary statistics are
     primarily associated with different valid-cell masks.
   - Confirmed temporal variability in rainfall.
   - LULC was not duplicated because its temporal analysis was already
     completed separately.

8. **Final predictor dataset summary**
   - Summarised the final modelling datasets for 2003, 2014, and 2025.
   - Confirmed predictor ranges, LULC classes, observation counts, and data
     integrity.

### Final modelling datasets

| Year | Observations |
|------|-------------:|
| 2003 | 47,523 |
| 2014 | 46,391 |
| 2025 | 43,076 |

Each year contains:

- 7 static continuous predictors
- 1 year-specific rainfall predictor
- 1 categorical LULC predictor
- spatial row and column indices
- UTM cell-centre coordinates
- year identifier

### Important diagnostic finding

Clay and Sand exhibited strong multicollinearity:

- Pearson correlation: approximately **r = 0.89**
- Static Clay VIF: approximately **7.05**
- Static Sand VIF: approximately **6.94**

These predictors were **not removed automatically**. Their treatment will be
determined later using statistical diagnostics, physical interpretation, and
the requirements of the selected modelling approaches.

### Final status

Notebook 07 provides a statistically and spatially documented predictor
dataset for the three modelling years.

No predictor values were modified, no missing values were artificially
imputed, and no predictor was removed during the notebook.

The resulting predictor datasets are ready for the next stage of the
flood-susceptibility workflow, where they will be integrated with the
flood-related target data.